En este notebook:

- Se calcula el riesgo cardivascular de las pacientes de los grupos mostrados en 'combinaciones con p valores - 6 mayo.pptx' con las calculadoras Framingham y Score2.
- Se calcula el mismo riesgo suponiendo que las pacientes son varones.

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Carga de archivos

carpeta = 'exportaciones_clustering'
archivos_csv = sorted([f for f in os.listdir(carpeta) if f.endswith('_pacientes.csv')])
print('Archivos de clusters encontrados:')
for f in archivos_csv:
    print(f'  {f}')

datasets = {}
for f in archivos_csv:
    key = f.replace('_pacientes.csv', '')
    datasets[key] = pd.read_csv(os.path.join(carpeta, f))
    datasets[key]['id'] = datasets[key]['id'].astype(float)
    print(f'  {key}: {len(datasets[key])} pacientes, '
          f'{datasets[key]["cluster"].nunique()} clusters')

Archivos de clusters encontrados:
  combo_3_k3_sil0.6052_pacientes.csv
  combo_7_k4_sil0.5821_pacientes.csv
  combo_8_k4_sil0.5743_pacientes.csv
  combo_3_k3_sil0.6052: 458 pacientes, 3 clusters
  combo_7_k4_sil0.5821: 458 pacientes, 4 clusters
  combo_8_k4_sil0.5743: 458 pacientes, 4 clusters


- Las variables necesarias en la fórmula Framingham son: edad, sexo, presión arterial sistólica, medicación para la hipertensión, colesterol total, hdl, fuma, diabetes.

- Las variables necesarias en la fórmula Score2 son: edad, sexo, presión arterial sistólica, fuma.

In [3]:
# Importar variables desde datos.csv

# datos.csv contiene las variables necesarias para las fórmulas de riesgo CV que no estaban en los exports de clustering:

#     edad_actual          → edad real en el momento del estudio (no gestacional)
#     ta_sistolica         → TAS postparto real (no proxy 1er trimestre)
#     colesterol_total     → mg/dL  (disponible en 69% de pacientes)
#     hdl                  → mg/dL  (disponible en 69%)
#     ldl                  → mg/dL  (disponible en 69%)
#     fuma_post            → 'Si' / 'No' / 'Ex_Fumadora'
#     diabetes_mellitus_1_2→ 0/1
#     hta_cronica          → 0/1
#     tomo_Antihipertensivo→ 0/1


df_datos = pd.read_csv('datos.csv')
df_datos2 = pd.read_csv('categoricas_nominales.csv')
df_datos['id'] = df_datos['id'].astype(float)

In [4]:
# Verificación de seguridad antes de asignar ids por posición
assert len(df_datos2) == len(df_datos), (
    f"ATENCIÓN: categoricas_nominales tiene {len(df_datos2)} filas "
    f"pero datos.csv tiene {len(df_datos)}. No se puede alinear por posición."
)

In [5]:
# Asignar id de datos.csv por posición de fila
df_datos2['id'] = df_datos['id'].values

In [6]:
# Añadir tomo_Antihipertensivo a df_cv para que llegue al merge posterior
df_datos = pd.merge(
    df_datos,
    df_datos2[['id', 'tomo_Antihipertensivo']],
    on='id',
    how='left'
)

In [7]:
# Variables: edad, colesterol_total (mg/dL), hdl (mg/dL), tas (mmHg), tabaco (0/1), diabetes (0/1), tratamiento_antihipertensivo (0/1)
cols_framingham = ['id', 'edad_actual', 'colesterol_total', 'hdl','ta_sistolica', 'fuma_post', 'diabetes_mellitus_1_2', 'tomo_Antihipertensivo']

cols_existentes = [c for c in cols_framingham if c in df_datos.columns]
df_cv = df_datos[cols_existentes].copy()

In [8]:
# Codificar la variable fuma_post

df_cv["fuma_post"] = df_cv["fuma_post"].map({"Si": 1, "Ex_Fumadora": 1, "No": 0})

In [9]:
# Merge con cada dataset de clusters
for key in datasets:
    df = datasets[key].copy()
    n_antes = len(df)

    # Columnas que ya existen en el export y también están en df_cv:
    # se mantiene la del export (no se sobreescribe), sufijo _cv para duplicados
    df = pd.merge(df, df_cv, on='id', how='left', suffixes=('', '_cv'))

    # Eliminar columnas duplicadas (_cv) — el export ya las tiene
    cols_dup = [c for c in df.columns if c.endswith('_cv')]
    df.drop(columns=cols_dup, inplace=True)

    assert len(df) == n_antes, f'Merge duplicó filas en {key}'
    datasets[key] = df

    n_col = df['colesterol_total'].notna().sum()
    n_hdl = df['hdl'].notna().sum()
    n_tas = df['ta_sistolica'].notna().sum()
    print(f'  {key}: merge OK — colesterol {n_col}/{len(df)}, '
          f'HDL {n_hdl}/{len(df)}, TAS postparto {n_tas}/{len(df)}')

  combo_3_k3_sil0.6052: merge OK — colesterol 315/458, HDL 315/458, TAS postparto 454/458
  combo_7_k4_sil0.5821: merge OK — colesterol 315/458, HDL 315/458, TAS postparto 454/458
  combo_8_k4_sil0.5743: merge OK — colesterol 315/458, HDL 315/458, TAS postparto 454/458


# K-Means

## Framingham (mujeres)

$$\text{Riesgo}_{10\text{ años}} = 1 - S_0(t)^{\exp(\beta_1X_1 + \beta_2X_2 + \dots + \beta_nX_n - \mu)}$$

In [10]:
def framingham_mujer(edad, colesterol_total, hdl, tas, fuma, diabetes, tomo_Antihipertensivo):
    """
    Riesgo Framingham a 10 años para mujeres (Wilson 1998).
    Devuelve (riesgo_pct, categoria) o (NaN, NaN) si faltan datos.
    """
    if any(pd.isna(v) for v in [edad, colesterol_total, hdl, tas, fuma, diabetes]):
        return np.nan, np.nan

    lnA = (2.32888  * np.log(edad)
         + 1.20904  * np.log(colesterol_total)
         - 0.70833  * np.log(hdl)
         + (2.76157 if tomo_Antihipertensivo else 2.82263) * np.log(tas)
         + 0.52873  * fuma
         + 0.69154  * diabetes
         - 26.0145)

    S0     = 0.95012
    riesgo = (1 - S0 ** np.exp(lnA)) * 100

    if   riesgo < 10: cat = 'Bajo (<10%)'
    elif riesgo < 20: cat = 'Moderado (10-20%)'
    else:             cat = 'Alto (>20%)'

    return round(riesgo, 2), cat

## Score2 (mujeres)

$$\text{Riesgo}_{10\text{ años}} = 1 - S_0(10)^{\exp(\text{PI} - \bar{\text{PI}})}$$

con $$\text{PI} = \beta_1 \cdot (\text{Edad} - 60) + \beta_2 \cdot \ln(\text{PAS}) + \beta_3 \cdot \ln(\text{Colesterol No-HDL}) + \beta_4 \cdot \text{Tabaco} + \text{Interacciones}$$


In [11]:
def score2_mujer(edad, colesterol_total, hdl, ta_sistolica, fuma):
    """
    SCORE2 a 10 años para mujeres, región riesgo bajo (España).
    Devuelve (riesgo_pct, categoria) o (NaN, NaN) si faltan datos.
    Categorías según umbrales ESC 2021 por franja de edad.
    """
    if any(pd.isna(v) for v in [edad, colesterol_total, hdl, ta_sistolica, fuma]):
        return np.nan, np.nan

    # Conversión mg/dL → mmol/L
    col_total_mmol = colesterol_total / 38.67
    hdl_mmol       = hdl      / 38.67
    col_no_hdl     = col_total_mmol - hdl_mmol

    # Centrado de variables (SCORE2, tabla S6)
    edad_c       = edad      - 60
    ta_sistolica_c        = (ta_sistolica      - 120) / 20
    col_no_hdl_c = col_no_hdl - 3.5

    # Suma lineal — coeficientes mujeres, riesgo bajo
    lp = (0.3524  * edad_c
        + 0.8564  * ta_sistolica_c
        + 0.8009  * fuma
        + 0.6322  * col_no_hdl_c
        - 0.0047  * edad_c * ta_sistolica_c
        - 0.0118  * edad_c * fuma
        - 0.0136  * edad_c * col_no_hdl_c
        - 0.3580)              # constante basal mujeres riesgo bajo

    S0     = 0.9776            # supervivencia basal mujeres riesgo bajo
    riesgo = (1 - S0 ** np.exp(lp)) * 100

    # Umbrales ESC 2021 según edad
    if edad < 50:
        if   riesgo < 2.5:  cat = 'Bajo (<2.5%)'
        elif riesgo < 7.5:  cat = 'Moderado (2.5-7.5%)'
        elif riesgo < 10:   cat = 'Alto (7.5-10%)'
        else:               cat = 'Muy alto (\u226510%)'
    else:
        if   riesgo < 5:    cat = 'Bajo (<5%)'
        elif riesgo < 10:   cat = 'Moderado (5-10%)'
        elif riesgo < 15:   cat = 'Alto (10-15%)'
        else:               cat = 'Muy alto (\u226515%)'

    return round(riesgo, 2), cat


## Aplicación y descripción

In [12]:
def aplicar_scores_cv(df):
    """Añade columnas de riesgo Framingham y SCORE2 a un DataFrame de pacientes."""
    fram_vals, fram_cats = [], []
    s2_vals,   s2_cats   = [], []

    for _, row in df.iterrows():
        edad      = row.get('edad_actual',         np.nan)
        col_total = row.get('colesterol_total',     np.nan)
        hdl       = row.get('hdl',                 np.nan)
        tas       = row.get('ta_sistolica',         np.nan)
        tabaco    = row.get('fuma_post',        np.nan)
        diabetes  = row.get('diabetes_mellitus_1_2',np.nan)
        antihta = row.get('tomo_Antihipertensivo', 0)

        fv, fc = framingham_mujer(edad, col_total, hdl, tas,
                                  tabaco, diabetes, antihta)
        sv, sc = score2_mujer(edad, col_total, hdl, tas, tabaco)

        fram_vals.append(fv);  fram_cats.append(fc)
        s2_vals.append(sv);    s2_cats.append(sc)

    df = df.copy()
    df['framingham_riesgo_pct'] = fram_vals
    df['framingham_categoria']  = fram_cats
    df['score2_riesgo_pct']     = s2_vals
    df['score2_categoria']      = s2_cats
    return df


resultados_finales = {}

for key, df in datasets.items():
    df_scored = aplicar_scores_cv(df)
    resultados_finales[key] = df_scored

    n       = len(df_scored)
    n_fram  = df_scored['framingham_riesgo_pct'].notna().sum()
    n_s2    = df_scored['score2_riesgo_pct'].notna().sum()
    print(f'\n{key}')
    print(f'  Total pacientes:        {n}')
    print(f'  Framingham calculable:  {n_fram} ({n_fram/n*100:.1f}%)')
    print(f'  SCORE2 calculable:      {n_s2}  ({n_s2/n*100:.1f}%)')
    if n_fram < n:
        missing = df_scored[
            df_scored['framingham_riesgo_pct'].isna()
        ][['id','colesterol_total','hdl','ta_sistolica']].head(3)
        print(f'  Pacientes incompletos (muestra):')
        print(missing.to_string(index=False))




combo_3_k3_sil0.6052
  Total pacientes:        458
  Framingham calculable:  313 (68.3%)
  SCORE2 calculable:      313  (68.3%)
  Pacientes incompletos (muestra):
   id  colesterol_total  hdl  ta_sistolica
182.0               NaN  NaN         152.0
221.0               NaN  NaN         108.0
289.0               NaN  NaN         120.0

combo_7_k4_sil0.5821
  Total pacientes:        458
  Framingham calculable:  313 (68.3%)
  SCORE2 calculable:      313  (68.3%)
  Pacientes incompletos (muestra):
   id  colesterol_total  hdl  ta_sistolica
 41.0               NaN  NaN          96.0
 68.0               NaN  NaN         101.0
141.0               NaN  NaN         112.0

combo_8_k4_sil0.5743
  Total pacientes:        458
  Framingham calculable:  313 (68.3%)
  SCORE2 calculable:      313  (68.3%)
  Pacientes incompletos (muestra):
   id  colesterol_total  hdl  ta_sistolica
 41.0               NaN  NaN          96.0
 79.0               NaN  NaN         115.0
182.0               NaN  NaN       

In [13]:
from IPython.core.display import HTML
from IPython.display import display

for key, df_scored in resultados_finales.items():

    print(f"\n{'='*65}")
    print(f"  {key}")
    print(f"{'='*65}")

    for score_col, cat_col, nombre in [
        ('framingham_riesgo_pct', 'framingham_categoria', 'FRAMINGHAM'),
        ('score2_riesgo_pct',     'score2_categoria',     'SCORE2'),
    ]:
        print(f"\n  ── {nombre} ──")

        # Tabla por paciente ───────────────────────────────────────
        df_pacientes = (
            df_scored[['id', 'cluster', score_col, cat_col]]
            .sort_values(['cluster', 'id'])
            .rename(columns={
                score_col: 'riesgo_10a (%)',
                cat_col:   'categoria',
            })
            .reset_index(drop=True)
        )
        df_pacientes['id'] = df_pacientes['id'].astype(int)
        display(HTML(
            '<div style="height:300px; overflow-y:scroll; border:1px solid #ccc; padding:4px">'
            + df_pacientes.to_html(index=False)
            + '</div>'
        ))

        # Resumen estadístico por cluster
        resumen = (
            df_scored
            .groupby('cluster')[score_col]
            .agg(
                N        = 'size',
                Media    = 'mean',
                Mediana  = 'median',
                SD       = 'std',
                Moda     = lambda x: round(x.dropna().mode().iloc[0], 2)
                           if not x.dropna().empty else np.nan,
                Min      = 'min',
                Max      = 'max',
            )
            .round(2)
        )
        # Categoría más frecuente por cluster
        cat_frecuente = (
            df_scored.dropna(subset=[cat_col])
            .groupby('cluster')[cat_col]
            .agg(lambda x: x.mode().iloc[0])
            .rename('Categoría más frecuente')
        )
        resumen = resumen.join(cat_frecuente)

        print(f"\n  Resumen por cluster:")
        display(resumen)


  combo_3_k3_sil0.6052

  ── FRAMINGHAM ──


id,cluster,riesgo_10a (%),categoria
32,0,3.13,Bajo (<10%)
33,0,3.88,Bajo (<10%)
42,0,5.07,Bajo (<10%)
57,0,5.35,Bajo (<10%)
75,0,4.88,Bajo (<10%)
86,0,10.75,Moderado (10-20%)
94,0,2.02,Bajo (<10%)
98,0,3.64,Bajo (<10%)
100,0,10.51,Moderado (10-20%)
107,0,10.13,Moderado (10-20%)



  Resumen por cluster:


,N,Media,Mediana,SD,Moda,Min,Max,Categoría más frecuente
cluster,,,,,,,,
0,112,5.70,4.49,4.91,3.88,0.50,36.33,Bajo (<10%)
1,34,6.07,5.28,4.26,1.47,1.47,18.82,Bajo (<10%)
2,312,3.65,2.87,2.92,1.92,0.62,24.90,Bajo (<10%)



  ── SCORE2 ──


...

,N,Media,Mediana,SD,Moda,Min,Max,Categoría más frecuente
cluster,,,,,,,,
0,91,4.83,3.68,3.81,1.92,1.37,23.25,Bajo (<10%)
1,91,5.88,4.50,5.32,3.88,0.50,36.33,Bajo (<10%)
2,24,4.76,3.74,3.13,0.88,0.88,14.71,Bajo (<10%)
3,252,3.52,2.82,2.76,2.19,0.62,24.90,Bajo (<10%)



  ── SCORE2 ──


id,cluster,riesgo_10a (%),categoria
2,0,0.01,Bajo (<2.5%)
5,0,0.00,Bajo (<2.5%)
8,0,0.00,Bajo (<2.5%)
13,0,0.00,Bajo (<2.5%)
14,0,0.00,Bajo (<2.5%)
18,0,0.00,Bajo (<2.5%)
19,0,0.01,Bajo (<2.5%)
41,0,NaN,NaN
42,0,0.00,Bajo (<2.5%)
43,0,0.02,Bajo (<2.5%)



  Resumen por cluster:


,N,Media,Mediana,SD,Moda,Min,Max,Categoría más frecuente
cluster,,,,,,,,
0,91,0.03,0.0,0.13,0.0,0.0,0.97,Bajo (<2.5%)
1,91,0.02,0.0,0.06,0.0,0.0,0.33,Bajo (<2.5%)
2,24,0.01,0.0,0.03,0.0,0.0,0.14,Bajo (<2.5%)
3,252,0.01,0.0,0.03,0.0,0.0,0.25,Bajo (<2.5%)


## Framingham (hombres)

$$\text{Riesgo}_{10\text{ años}} = 1 - S_0(t)^{\exp(\beta_1X_1 + \beta_2X_2 + \dots + \beta_nX_n - \mu)}$$

In [14]:
def framingham_hombre(edad, colesterol_total, hdl, tas, fuma, diabetes, tomo_Antihipertensivo):
    """
    Riesgo Framingham a 10 años para hombres (Wilson 1998).
    Devuelve (riesgo_pct, categoria) o (NaN, NaN) si faltan datos.
    """
    if any(pd.isna(v) for v in [edad, colesterol_total, hdl, tas, fuma, diabetes]):
        return np.nan, np.nan

    # Coeficientes oficiales para hombres (Wilson et al., 1998)
    lnA = (3.06117  * np.log(edad)
         + 1.12370  * np.log(colesterol_total)
         - 0.93261  * np.log(hdl)
         + (1.99881 if tomo_Antihipertensivo else 1.93303) * np.log(tas)
         + 0.65451  * fuma
         + 0.57367  * diabetes
         - 23.9802)

    # Supervivencia base a 10 años para hombres
    S0     = 0.88936
    riesgo = (1 - S0 ** np.exp(lnA)) * 100

    # Categorización del riesgo
    if   riesgo < 10: cat = 'Bajo (<10%)'
    elif riesgo < 20: cat = 'Moderado (10-20%)'
    else:             cat = 'Alto (>20%)'

    return round(riesgo, 2), cat

## Score2 (hombres)

$$\text{Riesgo}_{10\text{ años}} = 1 - S_0(10)^{\exp(\text{PI} - \bar{\text{PI}})}$$

con $$\text{PI} = \beta_1 \cdot (\text{Edad} - 60) + \beta_2 \cdot \ln(\text{PAS}) + \beta_3 \cdot \ln(\text{Colesterol No-HDL}) + \beta_4 \cdot \text{Tabaco} + \text{Interacciones}$$


In [15]:
def score2_hombre(edad, colesterol_total, hdl, ta_sistolica, fuma):
    """
    SCORE2 a 10 años para hombres, región riesgo bajo (España).
    Devuelve (riesgo_pct, categoria) o (NaN, NaN) si faltan datos.
    """
    if any(pd.isna(v) for v in [edad, colesterol_total, hdl, ta_sistolica, fuma]):
        return np.nan, np.nan

    # Conversión mg/dL → mmol/L
    col_total_mmol = colesterol_total / 38.67
    hdl_mmol       = hdl      / 38.67
    col_no_hdl     = col_total_mmol - hdl_mmol

    # Centrado de variables (SCORE2, idéntico para ambos sexos)
    edad_c        = edad - 60
    ta_sistolica_c = (ta_sistolica - 120) / 20
    col_no_hdl_c  = col_no_hdl - 3.5

    # Suma lineal — coeficientes HOMBRES, riesgo bajo (SCORE2 Working Group, 2021)
    lp = (0.2882  * edad_c
        + 0.7301  * ta_sistolica_c
        + 0.6385  * fuma
        + 0.5050  * col_no_hdl_c
        - 0.0039  * edad_c * ta_sistolica_c
        - 0.0076  * edad_c * fuma
        - 0.0094  * edad_c * col_no_hdl_c
        - 0.1470)              # Constante basal hombres riesgo bajo

    S0     = 0.9605            # Supervivencia basal hombres riesgo bajo
    riesgo = (1 - S0 ** np.exp(lp)) * 100

    # Umbrales ESC 2021 según edad (son iguales para hombres y mujeres)
    if edad < 50:
        if   riesgo < 2.5:  cat = 'Bajo (<2.5%)'
        elif riesgo < 7.5:  cat = 'Moderado (2.5-7.5%)'
        elif riesgo < 10:   cat = 'Alto (7.5-10%)'
        else:               cat = 'Muy alto (\u226510%)'
    else:
        if   riesgo < 5:    cat = 'Bajo (<5%)'
        elif riesgo < 10:   cat = 'Moderado (5-10%)'
        elif riesgo < 15:   cat = 'Alto (10-15%)'
        else:               cat = 'Muy alto (\u226515%)'

    return round(riesgo, 2), cat

## Aplicación y descripción

In [16]:
def aplicar_scores_cv_hombres(df):
    """Añade columnas de riesgo Framingham y SCORE2 para el escenario de hombres."""
    fram_h_vals, fram_h_cats = [], []
    s2_h_vals,   s2_h_cats   = [], []

    for _, row in df.iterrows():
        edad      = row.get('edad_actual',         np.nan)
        col_total = row.get('colesterol_total',     np.nan)
        hdl       = row.get('hdl',                 np.nan)
        tas       = row.get('ta_sistolica',         np.nan)
        tabaco    = row.get('fuma_post',            np.nan)
        diabetes  = row.get('diabetes_mellitus_1_2',np.nan)
        antihta   = row.get('tomo_Antihipertensivo', 0)

        # Usamos las funciones de hombres que definiste previamente
        fv, fc = framingham_hombre(edad, col_total, hdl, tas, tabaco, diabetes, antihta)
        sv, sc = score2_hombre(edad, col_total, hdl, tas, tabaco)

        fram_h_vals.append(fv);  fram_h_cats.append(fc)
        s2_h_vals.append(sv);    s2_h_cats.append(sc)

    df_res = df.copy()
    df_res['framingham_hombre_pct'] = fram_h_vals
    df_res['framingham_hombre_cat'] = fram_h_cats
    df_res['score2_hombre_pct']     = s2_h_vals
    df_res['score2_hombre_cat']     = s2_h_cats
    return df_res


resultados_finales_hombres = {}

for key, df in datasets.items():
    # CONTROL DE SEGURIDAD: Si las columnas clínicas se perdieron, hacemos el merge de nuevo
    if 'colesterol_total' not in df.columns:
        df = pd.merge(df, df_cv, on='id', how='left', suffixes=('', '_cv'))
        cols_dup = [c for c in df.columns if c.endswith('_cv')]
        df.drop(columns=cols_dup, inplace=True)

    # Calcular riesgos
    df_scored = aplicar_scores_cv_hombres(df)
    resultados_finales_hombres[key] = df_scored

    # Impresión de metadatos de control
    n       = len(df_scored)
    n_fram  = df_scored['framingham_hombre_pct'].notna().sum()
    n_s2    = df_scored['score2_hombre_pct'].notna().sum()
    
    print(f"\n" + "="*70)
    print(f" ARCHIVO: {key}")
    print(f"="*70)
    print(f' Total pacientes:        {n}')
    print(f' Framingham calculable:  {n_fram} ({n_fram/n*100:.1f}%)')
    print(f' SCORE2 calculable:      {n_s2}  ({n_s2/n*100:.1f}%)')
    
    if n_fram < n:
        missing = df_scored[
            df_scored['framingham_hombre_pct'].isna()
        ][['id','colesterol_total','hdl','ta_sistolica']].head(3)
        print(f' Pacientes incompletos (muestra):')
        print(missing.to_string(index=False))

    # DESPLIEGUE DE TABLAS DETALLADAS POR PACIENTE Y CLUSTER
    for score_col, cat_col, nombre_algoritmo in [
        ('framingham_hombre_pct', 'framingham_hombre_cat', 'FRAMINGHAM (ESCENARIO HOMBRES)'),
        ('score2_hombre_pct',     'score2_hombre_cat',     'SCORE2 (ESCENARIO HOMBRES)'),
    ]:
        print(f"\n ── {nombre_algoritmo} ──")

        # Construcción de la tabla detallada ordenada por Cluster e ID de paciente
        df_pacientes = (
            df_scored[['id', 'cluster', score_col, cat_col]]
            .sort_values(['cluster', 'id'])
            .rename(columns={
                score_col: 'riesgo_10a (%)',
                cat_col:   'categoria',
            })
            .reset_index(drop=True)
        )
        df_pacientes['id'] = df_pacientes['id'].astype(int)
        
        # Mostramos la tabla completa de pacientes en el Jupyter Notebook
        display(HTML(
            '<div style="height:300px; overflow-y:scroll; border:1px solid #ccc; padding:4px">'
            + df_pacientes.to_html(index=False)
            + '</div>'
        ))

        # Resumen estadístico complementario para ver el comportamiento del grupo
        resumen = (
            df_scored
            .groupby('cluster')[score_col]
            .agg(
                N        = 'size',
                Media    = 'mean',
                Mediana  = 'median',
                SD       = 'std',
                Min      = 'min',
                Max      = 'max',
            )
            .round(2)
        )
        
        # Categoría de riesgo predominante en el cluster de hombres
        cat_frecuente = (
            df_scored.dropna(subset=[cat_col])
            .groupby('cluster')[cat_col]
            .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
            .rename('Categoría más frecuente')
        )
        resumen = resumen.join(cat_frecuente)

        print(f" Resumen estadístico de hombres por cluster:")
        display(resumen)


 ARCHIVO: combo_3_k3_sil0.6052
 Total pacientes:        458
 Framingham calculable:  313 (68.3%)
 SCORE2 calculable:      313  (68.3%)
 Pacientes incompletos (muestra):
   id  colesterol_total  hdl  ta_sistolica
182.0               NaN  NaN         152.0
221.0               NaN  NaN         108.0
289.0               NaN  NaN         120.0

 ── FRAMINGHAM (ESCENARIO HOMBRES) ──


id,cluster,riesgo_10a (%),categoria
32,0,2.96,Bajo (<10%)
33,0,4.30,Bajo (<10%)
42,0,4.71,Bajo (<10%)
57,0,6.80,Bajo (<10%)
75,0,4.16,Bajo (<10%)
86,0,11.45,Moderado (10-20%)
94,0,1.68,Bajo (<10%)
98,0,3.99,Bajo (<10%)
100,0,10.68,Moderado (10-20%)
107,0,11.76,Moderado (10-20%)


 Resumen estadístico de hombres por cluster:


,N,Media,Mediana,SD,Min,Max,Categoría más frecuente
cluster,,,,,,,
0,112,5.74,4.34,4.30,0.31,26.74,Bajo (<10%)
1,34,6.52,4.94,4.73,1.24,19.58,Bajo (<10%)
2,312,3.98,3.02,3.15,0.46,24.64,Bajo (<10%)



 ── SCORE2 (ESCENARIO HOMBRES) ──


...

,N,Media,Mediana,SD,Min,Max,Categoría más frecuente
cluster,,,,,,,
0,91,5.19,3.99,4.14,1.28,24.64,Bajo (<10%)
1,91,5.96,4.35,4.71,0.31,26.74,Bajo (<10%)
2,24,4.78,3.69,2.79,1.07,11.67,Bajo (<10%)
3,252,3.86,3.01,2.98,0.46,22.63,Bajo (<10%)



 ── SCORE2 (ESCENARIO HOMBRES) ──


id,cluster,riesgo_10a (%),categoria
2,0,0.04,Bajo (<2.5%)
5,0,0.00,Bajo (<2.5%)
8,0,0.01,Bajo (<2.5%)
13,0,0.02,Bajo (<2.5%)
14,0,0.02,Bajo (<2.5%)
18,0,0.00,Bajo (<2.5%)
19,0,0.03,Bajo (<2.5%)
41,0,NaN,NaN
42,0,0.01,Bajo (<2.5%)
43,0,0.10,Bajo (<2.5%)


 Resumen estadístico de hombres por cluster:


,N,Media,Mediana,SD,Min,Max,Categoría más frecuente
cluster,,,,,,,
0,91,0.10,0.02,0.31,0.0,2.20,Bajo (<2.5%)
1,91,0.07,0.02,0.17,0.0,1.04,Bajo (<2.5%)
2,24,0.06,0.02,0.11,0.0,0.53,Bajo (<2.5%)
3,252,0.04,0.01,0.09,0.0,0.73,Bajo (<2.5%)


# K-Medoids

In [17]:
carpeta_km = 'exportaciones_kmedoids'
archivos_csv_km = sorted([f for f in os.listdir(carpeta_km) if f.endswith('_pacientes.csv')])
print('Archivos de clusters K-Medoids encontrados:')
for f in archivos_csv_km:
    print(f'  {f}')

datasets_km = {}
for f in archivos_csv_km:
    key = f.replace('_pacientes.csv', '')
    datasets_km[key] = pd.read_csv(os.path.join(carpeta_km, f))
    datasets_km[key]['id'] = datasets_km[key]['id'].astype(float)
    print(f'  {key}: {len(datasets_km[key])} pacientes, '
          f'{datasets_km[key]["cluster"].nunique()} clusters')

# MERGE CON VARIABLES CLÍNICAS (df_cv ya construido en la sección K-Means)

for key in datasets_km:
    df = datasets_km[key].copy()
    n_antes = len(df)

    df = pd.merge(df, df_cv, on='id', how='left', suffixes=('', '_cv'))
    cols_dup = [c for c in df.columns if c.endswith('_cv')]
    df.drop(columns=cols_dup, inplace=True)

    assert len(df) == n_antes, f'Merge duplicó filas en {key}'
    datasets_km[key] = df

    n_col = df['colesterol_total'].notna().sum()
    n_hdl = df['hdl'].notna().sum()
    n_tas = df['ta_sistolica'].notna().sum()
    print(f'  {key}: merge OK — colesterol {n_col}/{len(df)}, '
          f'HDL {n_hdl}/{len(df)}, TAS postparto {n_tas}/{len(df)}')

Archivos de clusters K-Medoids encontrados:
  combo_1_k2_sil0.9740_pacientes.csv
  combo_2_k2_sil0.9637_pacientes.csv
  combo_1_k2_sil0.9740: 458 pacientes, 2 clusters
  combo_2_k2_sil0.9637: 458 pacientes, 2 clusters
  combo_1_k2_sil0.9740: merge OK — colesterol 315/458, HDL 315/458, TAS postparto 454/458
  combo_2_k2_sil0.9637: merge OK — colesterol 315/458, HDL 315/458, TAS postparto 454/458


## Framingham y SCORE2 (mujeres)

In [18]:
resultados_km_mujeres = {}

for key, df in datasets_km.items():
    df_scored = aplicar_scores_cv(df)
    resultados_km_mujeres[key] = df_scored

    n      = len(df_scored)
    n_fram = df_scored['framingham_riesgo_pct'].notna().sum()
    n_s2   = df_scored['score2_riesgo_pct'].notna().sum()
    print(f'\n{key}')
    print(f'  Total pacientes:        {n}')
    print(f'  Framingham calculable:  {n_fram} ({n_fram/n*100:.1f}%)')
    print(f'  SCORE2 calculable:      {n_s2}  ({n_s2/n*100:.1f}%)')
    if n_fram < n:
        missing = df_scored[
            df_scored['framingham_riesgo_pct'].isna()
        ][['id', 'colesterol_total', 'hdl', 'ta_sistolica']].head(3)
        print(f'  Pacientes incompletos (muestra):')
        print(missing.to_string(index=False))

for key, df_scored in resultados_km_mujeres.items():

    print(f"\n{'='*65}")
    print(f"  {key}")
    print(f"{'='*65}")

    for score_col, cat_col, nombre in [
        ('framingham_riesgo_pct', 'framingham_categoria', 'FRAMINGHAM'),
        ('score2_riesgo_pct',     'score2_categoria',     'SCORE2'),
    ]:
        print(f"\n  ── {nombre} ──")

        df_pacientes = (
            df_scored[['id', 'cluster', score_col, cat_col]]
            .sort_values(['cluster', 'id'])
            .rename(columns={
                score_col: 'riesgo_10a (%)',
                cat_col:   'categoria',
            })
            .reset_index(drop=True)
        )
        df_pacientes['id'] = df_pacientes['id'].astype(int)
        display(HTML(
            '<div style="height:300px; overflow-y:scroll; border:1px solid #ccc; padding:4px">'
            + df_pacientes.to_html(index=False)
            + '</div>'
        ))

        resumen = (
            df_scored
            .groupby('cluster')[score_col]
            .agg(
                N       = 'size',
                Media   = 'mean',
                Mediana = 'median',
                SD      = 'std',
                Moda    = lambda x: round(x.dropna().mode().iloc[0], 2)
                          if not x.dropna().empty else np.nan,
                Min     = 'min',
                Max     = 'max',
            )
            .round(2)
        )
        cat_frecuente = (
            df_scored.dropna(subset=[cat_col])
            .groupby('cluster')[cat_col]
            .agg(lambda x: x.mode().iloc[0])
            .rename('Categoría más frecuente')
        )
        resumen = resumen.join(cat_frecuente)

        print(f"\n  Resumen por cluster:")
        display(resumen)


combo_1_k2_sil0.9740
  Total pacientes:        458
  Framingham calculable:  313 (68.3%)
  SCORE2 calculable:      313  (68.3%)
  Pacientes incompletos (muestra):
   id  colesterol_total  hdl  ta_sistolica
182.0               NaN  NaN         152.0
283.0               NaN  NaN         135.0
289.0               NaN  NaN         120.0

combo_2_k2_sil0.9637
  Total pacientes:        458
  Framingham calculable:  313 (68.3%)
  SCORE2 calculable:      313  (68.3%)
  Pacientes incompletos (muestra):
  id  colesterol_total  hdl  ta_sistolica
41.0               NaN  NaN          96.0
68.0               NaN  NaN         101.0
79.0               NaN  NaN         115.0

  combo_1_k2_sil0.9740

  ── FRAMINGHAM ──


id,cluster,riesgo_10a (%),categoria
1,0,1.70,Bajo (<10%)
2,0,2.39,Bajo (<10%)
4,0,4.80,Bajo (<10%)
13,0,4.19,Bajo (<10%)
32,0,3.13,Bajo (<10%)
33,0,3.88,Bajo (<10%)
39,0,6.15,Bajo (<10%)
42,0,5.07,Bajo (<10%)
53,0,3.23,Bajo (<10%)
56,0,5.70,Bajo (<10%)



  Resumen por cluster:


,N,Media,Mediana,SD,Moda,Min,Max,Categoría más frecuente
cluster,,,,,,,,
0,132,5.26,4.47,3.70,1.59,0.50,24.90,Bajo (<10%)
1,326,3.96,2.92,3.69,1.92,0.62,36.33,Bajo (<10%)



  ── SCORE2 ──


...

,N,Media,Mediana,SD,Moda,Min,Max,Categoría más frecuente
cluster,,,,,,,,
0,326,3.96,2.92,3.69,1.92,0.62,36.33,Bajo (<10%)
1,132,5.26,4.47,3.70,1.59,0.50,24.90,Bajo (<10%)



  ── SCORE2 ──


id,cluster,riesgo_10a (%),categoria
5,0,0.00,Bajo (<2.5%)
8,0,0.00,Bajo (<2.5%)
10,0,0.00,Bajo (<2.5%)
11,0,0.00,Bajo (<2.5%)
14,0,0.00,Bajo (<2.5%)
15,0,0.00,Bajo (<2.5%)
18,0,0.00,Bajo (<2.5%)
19,0,0.01,Bajo (<2.5%)
26,0,0.00,Bajo (<2.5%)
28,0,0.01,Bajo (<2.5%)



  Resumen por cluster:


,N,Media,Mediana,SD,Moda,Min,Max,Categoría más frecuente
cluster,,,,,,,,
0,326,0.01,0.0,0.04,0.0,0.0,0.33,Bajo (<2.5%)
1,132,0.03,0.0,0.11,0.0,0.0,0.97,Bajo (<2.5%)


## Framingham y SCORE2 (hombres)

In [19]:
resultados_km_hombres = {}

for key, df in datasets_km.items():
    if 'colesterol_total' not in df.columns:
        df = pd.merge(df, df_cv, on='id', how='left', suffixes=('', '_cv'))
        cols_dup = [c for c in df.columns if c.endswith('_cv')]
        df.drop(columns=cols_dup, inplace=True)

    df_scored = aplicar_scores_cv_hombres(df)
    resultados_km_hombres[key] = df_scored

    n      = len(df_scored)
    n_fram = df_scored['framingham_hombre_pct'].notna().sum()
    n_s2   = df_scored['score2_hombre_pct'].notna().sum()

    print(f"\n" + "="*70)
    print(f" ARCHIVO: {key}")
    print(f"="*70)
    print(f' Total pacientes:        {n}')
    print(f' Framingham calculable:  {n_fram} ({n_fram/n*100:.1f}%)')
    print(f' SCORE2 calculable:      {n_s2}  ({n_s2/n*100:.1f}%)')

    if n_fram < n:
        missing = df_scored[
            df_scored['framingham_hombre_pct'].isna()
        ][['id', 'colesterol_total', 'hdl', 'ta_sistolica']].head(3)
        print(f' Pacientes incompletos (muestra):')
        print(missing.to_string(index=False))

    for score_col, cat_col, nombre_algoritmo in [
        ('framingham_hombre_pct', 'framingham_hombre_cat', 'FRAMINGHAM (ESCENARIO HOMBRES)'),
        ('score2_hombre_pct',     'score2_hombre_cat',     'SCORE2 (ESCENARIO HOMBRES)'),
    ]:
        print(f"\n ── {nombre_algoritmo} ──")

        df_pacientes = (
            df_scored[['id', 'cluster', score_col, cat_col]]
            .sort_values(['cluster', 'id'])
            .rename(columns={
                score_col: 'riesgo_10a (%)',
                cat_col:   'categoria',
            })
            .reset_index(drop=True)
        )
        df_pacientes['id'] = df_pacientes['id'].astype(int)
        display(HTML(
            '<div style="height:300px; overflow-y:scroll; border:1px solid #ccc; padding:4px">'
            + df_pacientes.to_html(index=False)
            + '</div>'
        ))

        resumen = (
            df_scored
            .groupby('cluster')[score_col]
            .agg(
                N       = 'size',
                Media   = 'mean',
                Mediana = 'median',
                SD      = 'std',
                Min     = 'min',
                Max     = 'max',
            )
            .round(2)
        )
        cat_frecuente = (
            df_scored.dropna(subset=[cat_col])
            .groupby('cluster')[cat_col]
            .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
            .rename('Categoría más frecuente')
        )
        resumen = resumen.join(cat_frecuente)

        print(f" Resumen estadístico de hombres por cluster:")
        display(resumen)


 ARCHIVO: combo_1_k2_sil0.9740
 Total pacientes:        458
 Framingham calculable:  313 (68.3%)
 SCORE2 calculable:      313  (68.3%)
 Pacientes incompletos (muestra):
   id  colesterol_total  hdl  ta_sistolica
182.0               NaN  NaN         152.0
283.0               NaN  NaN         135.0
289.0               NaN  NaN         120.0

 ── FRAMINGHAM (ESCENARIO HOMBRES) ──


id,cluster,riesgo_10a (%),categoria
1,0,1.64,Bajo (<10%)
2,0,3.14,Bajo (<10%)
4,0,8.77,Bajo (<10%)
13,0,4.54,Bajo (<10%)
32,0,2.96,Bajo (<10%)
33,0,4.30,Bajo (<10%)
39,0,6.46,Bajo (<10%)
42,0,4.71,Bajo (<10%)
53,0,3.64,Bajo (<10%)
56,0,5.57,Bajo (<10%)


 Resumen estadístico de hombres por cluster:


,N,Media,Mediana,SD,Min,Max,Categoría más frecuente
cluster,,,,,,,
0,132,5.58,4.40,4.07,0.31,22.63,Bajo (<10%)
1,326,4.21,3.18,3.47,0.46,26.74,Bajo (<10%)



 ── SCORE2 (ESCENARIO HOMBRES) ──


...

,N,Media,Mediana,SD,Min,Max,Categoría más frecuente
cluster,,,,,,,
0,326,4.21,3.18,3.47,0.46,26.74,Bajo (<10%)
1,132,5.58,4.40,4.07,0.31,22.63,Bajo (<10%)



 ── SCORE2 (ESCENARIO HOMBRES) ──


id,cluster,riesgo_10a (%),categoria
5,0,0.00,Bajo (<2.5%)
8,0,0.01,Bajo (<2.5%)
10,0,0.02,Bajo (<2.5%)
11,0,0.00,Bajo (<2.5%)
14,0,0.02,Bajo (<2.5%)
15,0,0.00,Bajo (<2.5%)
18,0,0.00,Bajo (<2.5%)
19,0,0.03,Bajo (<2.5%)
26,0,0.02,Bajo (<2.5%)
28,0,0.06,Bajo (<2.5%)


 Resumen estadístico de hombres por cluster:


,N,Media,Mediana,SD,Min,Max,Categoría más frecuente
cluster,,,,,,,
0,326,0.05,0.01,0.13,0.0,1.04,Bajo (<2.5%)
1,132,0.09,0.02,0.26,0.0,2.20,Bajo (<2.5%)


# Análisis de missings

COMBINACIÓN 1: K-MEDOIDS con Silhouette Score 0.974
- Cluster 0: 42 missings de 132
- Cluster 1: 102 missings de 326

COMBINACIÓN 2: K-MEDOIDS con Silhouette Score 0.9637
- Cluster 0: 102 missings de 326
- Cluster 1: 42 missings de 132

COMBINACIÓN 3: K-MEANS con Silhouette Score 0.6052
- Cluster 0: 36 missings de 112
- Cluster 1: 10 missings de 34
- Cluster 2: 99 missings de 312

COMBINACIÓN 4: K-MEANS con Silhouette Score 0.5821
- Cluster 0: 84 missings de 268
- Cluster 1: 27 missings de 87
- Cluster 2: 24 missings de 71
- Cluster 3: 10 missings de 32

COMBINACIÓN 5: K-MEANS con Silhouette Score 0.5743
- Cluster 0: 26 missings de 91
- Cluster 1: 30 missings de 91
- Cluster 2: 3 missings de 24
- Cluster 3: 86 missings de 252

In [20]:
df_datos['fuma_post'].isnull().sum()

np.int64(0)

In [21]:
df_datos['diabetes_mellitus_1_2'].isnull().sum()

np.int64(0)

In [22]:
df_datos['edad_actual'].isnull().sum()

np.int64(0)

In [23]:
df_datos['hdl'].isnull().sum()

np.int64(144)

In [24]:
df_datos['colesterol_total'].isnull().sum()

np.int64(144)

In [25]:
df_datos['ta_sistolica'].isnull().sum()

np.int64(4)

# Ídem con las combinaciones de k-Medoids

In [ ]:
# CARGA DE ARCHIVOS DE K-MEDOIDS
carpeta_kmedoids = 'exportaciones_kmedoids'
archivos_csv = sorted([f for f in os.listdir(carpeta_kmedoids) if f.endswith('_pacientes.csv')])

print('Archivos de clusters (K-Medoids) encontrados:')
for f in archivos_csv:
    print(f'  {f}')

datasets = {}
for f in archivos_csv:
    key = f.replace('_pacientes.csv', '')
    datasets[key] = pd.read_csv(os.path.join(carpeta_kmedoids, f))
    datasets[key]['id'] = datasets[key]['id'].astype(float)
    print(f'  {key}: {len(datasets[key])} pacientes, '
          f'{datasets[key]["cluster"].nunique()} clusters')


# IMPORTAR VARIABLES CLÍNICAS DESDE BASE
df_datos = pd.read_csv('datos.csv')
df_datos2 = pd.read_csv('categoricas_nominales.csv')
df_datos['id'] = df_datos['id'].astype(float)

# Verificación de seguridad y alineación de ids por posición
assert len(df_datos2) == len(df_datos), (
    f"ATENCIÓN: categoricas_nominales tiene {len(df_datos2)} filas "
    f"pero datos.csv tiene {len(df_datos)}. No se puede alinear por posición."
)
df_datos2['id'] = df_datos['id'].values

# Cruzar con variables adicionales requeridas
df_datos = pd.merge(
    df_datos,
    df_datos2[['id', 'tomo_Antihipertensivo']],
    on='id',
    how='left'
)

# Filtrar las columnas necesarias para el cálculo cardiovascular
cols_framingham = ['id', 'edad_actual', 'colesterol_total', 'hdl', 'ta_sistolica', 'fuma_post', 'diabetes_mellitus_1_2', 'tomo_Antihipertensivo']
cols_existentes = [c for c in cols_framingham if c in df_datos.columns]
df_cv = df_datos[cols_existentes].copy()

# Codificar la variable fuma_post de manera idéntica al notebook original
df_cv["fuma_post"] = df_cv["fuma_post"].map({"Si": 1, "Ex_Fumadora": 1, "No": 0})


# MERGE CLÍNICO CON DATASETS K-MEDOIDS
for key in datasets:
    df = datasets[key].copy()
    n_antes = len(df)

    # Se mantiene la variable del export original si ya existe, sufijo _cv para duplicadas
    df = pd.merge(df, df_cv, on='id', how='left', suffixes=('', '_cv'))

    # Eliminar duplicados generados por el merge clínico
    cols_dup = [c for c in df.columns if c.endswith('_cv')]
    df.drop(columns=cols_dup, inplace=True)

    assert len(df) == n_antes, f'Merge duplicó filas en {key}'
    datasets[key] = df

    n_col = df['colesterol_total'].notna().sum()
    n_hdl = df['hdl'].notna().sum()
    n_tas = df['ta_sistolica'].notna().sum()
    print(f'  {key}: merge OK — colesterol {n_col}/{len(df)}, '
          f'HDL {n_hdl}/{len(df)}, TAS postparto {n_tas}/{len(df)}')


# DEFINICIÓN DE FÓRMULAS PARA MUJERES
def framingham_mujer(edad, colesterol_total, hdl, tas, fuma, diabetes, tomo_Antihipertensivo):
    if any(pd.isna(v) for v in [edad, colesterol_total, hdl, tas, fuma, diabetes]):
        return np.nan, np.nan

    lnA = (2.32888  * np.log(edad)
         + 1.20904  * np.log(colesterol_total)
         - 0.70833  * np.log(hdl)
         + (2.76157 if tomo_Antihipertensivo else 2.82263) * np.log(tas)
         + 0.52873  * fuma
         + 0.69154  * diabetes
         - 26.0145)

    S0     = 0.95012
    riesgo = (1 - S0 ** np.exp(lnA)) * 100

    if   riesgo < 10: cat = 'Bajo (<10%)'
    elif riesgo < 20: cat = 'Moderado (10-20%)'
    else:             cat = 'Alto (>20%)'

    return round(riesgo, 2), cat

def score2_mujer(edad, colesterol_total, hdl, ta_sistolica, fuma):
    if any(pd.isna(v) for v in [edad, colesterol_total, hdl, ta_sistolica, fuma]):
        return np.nan, np.nan

    col_total_mmol = colesterol_total / 38.67
    hdl_mmol       = hdl      / 38.67
    col_no_hdl     = col_total_mmol - hdl_mmol

    edad_c       = edad      - 60
    ta_sistolica_c        = (ta_sistolica      - 120) / 20
    col_no_hdl_c = col_no_hdl - 3.5

    lp = (0.3524  * edad_c
        + 0.8564  * ta_sistolica_c
        + 0.8009  * fuma
        + 0.6322  * col_no_hdl_c
        - 0.0047  * edad_c * ta_sistolica_c
        - 0.0118  * edad_c * fuma
        - 0.0136  * edad_c * col_no_hdl_c
        - 0.3580)

    S0     = 0.9776
    riesgo = (1 - S0 ** np.exp(lp)) * 100

    if edad < 50:
        if   riesgo < 2.5:  cat = 'Bajo (<2.5%)'
        elif riesgo < 7.5:  cat = 'Moderado (2.5-7.5%)'
        elif riesgo < 10:   cat = 'Alto (7.5-10%)'
        else:               cat = 'Muy alto (≥10%)'
    else:
        if   riesgo < 5:    cat = 'Bajo (<5%)'
        elif riesgo < 10:   cat = 'Moderado (5-10%)'
        elif riesgo < 15:   cat = 'Alto (10-15%)'
        else:               cat = 'Muy alto (≥15%)'

    return round(riesgo, 2), cat

# DEFINICIÓN DE FÓRMULAS PARA HOMBRES
def framingham_hombre(edad, colesterol_total, hdl, tas, fuma, diabetes, tomo_Antihipertensivo):
    if any(pd.isna(v) for v in [edad, colesterol_total, hdl, tas, fuma, diabetes]):
        return np.nan, np.nan
    
    lnA = (3.06117  * np.log(edad)
         + 1.12370  * np.log(colesterol_total)
         - 0.93263  * np.log(hdl)
         + (1.99881 if tomo_Antihipertensivo else 1.93303) * np.log(tas)
         + 0.65451  * fuma
         + 0.57367  * diabetes
         - 23.9802)
    
    S0     = 0.88936
    riesgo = (1 - S0 ** np.exp(lnA)) * 100
    
    if   riesgo < 10: cat = 'Bajo (<10%)'
    elif riesgo < 20: cat = 'Moderado (10-20%)'
    else:             cat = 'Alto (>20%)'
    return round(riesgo, 2), cat

def score2_hombre(edad, colesterol_total, hdl, ta_sistolica, fuma):
    if any(pd.isna(v) for v in [edad, colesterol_total, hdl, ta_sistolica, fuma]):
        return np.nan, np.nan
    
    col_total_mmol = colesterol_total / 38.67
    hdl_mmol       = hdl      / 38.67
    col_no_hdl     = col_total_mmol - hdl_mmol
    
    edad_c       = edad      - 60
    ta_sistolica_c        = (ta_sistolica      - 120) / 20
    col_no_hdl_c = col_no_hdl - 3.5
    
    lp = (0.3742  * edad_c
        + 0.6053  * ta_sistolica_c
        + 0.5290  * fuma
        + 0.3879  * col_no_hdl_c
        - 0.0019  * edad_c * ta_sistolica_c
        - 0.0089  * edad_c * fuma
        - 0.0028  * edad_c * col_no_hdl_c
        - 0.5699)
    
    S0     = 0.9485
    riesgo = (1 - S0 ** np.exp(lp)) * 100
    
    if edad < 50:
        if   riesgo < 2.5:  cat = 'Bajo (<2.5%)'
        elif riesgo < 7.5:  cat = 'Moderado (2.5-7.5%)'
        elif riesgo < 10:   cat = 'Alto (7.5-10%)'
        else:               cat = 'Muy alto (≥10%)'
    else:
        if   riesgo < 5:    cat = 'Bajo (<5%)'
        elif riesgo < 10:   cat = 'Moderado (5-10%)'
        elif riesgo < 15:   cat = 'Alto (10-15%)'
        else:               cat = 'Muy alto (≥15%)'
    return round(riesgo, 2), cat

resultados_finales = {}

for key, df in datasets.items():
    df_scored = aplicar_scores_cv(df)
    resultados_finales[key] = df_scored

    n       = len(df_scored)
    n_fram  = df_scored['framingham_riesgo_pct'].notna().sum()
    n_s2    = df_scored['score2_riesgo_pct'].notna().sum()
    print(f'\n{key}')
    print(f'  Total pacientes:        {n}')
    print(f'  Framingham calculable:  {n_fram} ({n_fram/n*100:.1f}%)')
    print(f'  SCORE2 calculable:      {n_s2}  ({n_s2/n*100:.1f}%)')
    if n_fram < n:
        missing = df_scored[
            df_scored['framingham_riesgo_pct'].isna()
        ][['id','colesterol_total','hdl','ta_sistolica']].head(3)
        print(f'  Pacientes incompletos (muestra):')
        print(missing.to_string(index=False))

Archivos de clusters (K-Medoids) encontrados:
  combo_1_k2_sil0.9740_pacientes.csv
  combo_2_k2_sil0.9637_pacientes.csv
  combo_1_k2_sil0.9740: 458 pacientes, 2 clusters
  combo_2_k2_sil0.9637: 458 pacientes, 2 clusters
  combo_1_k2_sil0.9740: merge OK — colesterol 315/458, HDL 315/458, TAS postparto 454/458
  combo_2_k2_sil0.9637: merge OK — colesterol 315/458, HDL 315/458, TAS postparto 454/458

combo_1_k2_sil0.9740
  Total pacientes:        458
  Framingham calculable:  313 (68.3%)
  SCORE2 calculable:      313  (68.3%)
  Pacientes incompletos (muestra):
   id  colesterol_total  hdl  ta_sistolica
182.0               NaN  NaN         152.0
283.0               NaN  NaN         135.0
289.0               NaN  NaN         120.0

combo_2_k2_sil0.9637
  Total pacientes:        458
  Framingham calculable:  313 (68.3%)
  SCORE2 calculable:      313  (68.3%)
  Pacientes incompletos (muestra):
  id  colesterol_total  hdl  ta_sistolica
41.0               NaN  NaN          96.0
68.0            

In [32]:
import os
from IPython.display import display, HTML

# CÁLCULO COMPLETO EN AMBOS SEXOS
resultados_completos = {}

for key, df in datasets.items():
    df_res = df.copy()
    
    # Listas intermedias para almacenar resultados
    f_h_v, f_h_c, s2_h_v, s2_h_c = [], [], [], []
    f_m_v, f_m_c, s2_m_v, s2_m_c = [], [], [], []
    
    for _, row in df_res.iterrows():
        # Variables necesarias
        ed   = row.get('edad_actual', np.nan)
        col  = row.get('colesterol_total', np.nan)
        hdl  = row.get('hdl', np.nan)
        tas  = row.get('ta_sistolica', np.nan)
        tab  = row.get('fuma_post', np.nan)
        diab = row.get('diabetes_mellitus_1_2', np.nan)
        anti = row.get('tomo_Antihipertensivo', 0)
        
        # 1. Simulación Mujeres (Original)
        fv_m, fc_m = framingham_mujer(ed, col, hdl, tas, tab, diab, anti)
        sv_m, sc_m = score2_mujer(ed, col, hdl, tas, tab)
        
        # 2. Simulación Hombres
        fv_h, fc_h = framingham_hombre(ed, col, hdl, tas, tab, diab, anti)
        sv_h, sc_h = score2_hombre(ed, col, hdl, tas, tab)
        
        f_m_v.append(fv_m); f_m_c.append(fc_m); s2_m_v.append(sv_m); s2_m_c.append(sc_m)
        f_h_v.append(fv_h); f_h_c.append(fc_h); s2_h_v.append(sv_h); s2_h_c.append(sc_h)
    
    # Asignación de columnas al DataFrame resultante
    df_res['Framingham_Mujer_%'] = f_m_v
    df_res['Framingham_Mujer_Cat'] = f_m_c
    df_res['SCORE2_Mujer_%'] = s2_m_v
    df_res['SCORE2_Mujer_Cat'] = s2_m_c
    
    df_res['Framingham_Hombre_%'] = f_h_v
    df_res['Framingham_Hombre_Cat'] = f_h_c
    df_res['SCORE2_Hombre_%'] = s2_h_v
    df_res['SCORE2_Hombre_Cat'] = s2_h_c
    
    resultados_completos[key] = df_res

# RESUMEN ESTADÍSTICO POR CLÚSTER

# Definimos los scores numéricos que queremos resumir
columnas_scores_num = [
    'Framingham_Mujer_%', 'SCORE2_Mujer_%',
    'Framingham_Hombre_%', 'SCORE2_Hombre_%'
]

# Definimos las columnas categóricas de riesgo
columnas_scores_cat = [
    'Framingham_Mujer_Cat', 'SCORE2_Mujer_Cat',
    'Framingham_Hombre_Cat', 'SCORE2_Hombre_Cat'
]

for key, df_scored in resultados_completos.items():
    display(HTML(f"<h2 style='color:#d9534f; margin-top:40px; border-bottom: 2px solid #d9534f;'> RESUMEN ESTADÍSTICO: {key.upper()}</h2>"))
    
    clusters_unicos = sorted(df_scored['cluster'].unique())
    
    for clus in clusters_unicos:
        df_cluster = df_scored[df_scored['cluster'] == clus]
        
        display(HTML(f"<h3><b>Análisis del Clúster {clus}</b> (n = {len(df_cluster)} pacientes)</h3>"))
        
        # 4.1. Métricas Numéricas (Media, Mediana, Desviación, etc.)
        # Calculamos los estadísticos principales transponiendo para que quede estético
        resumen_numerico = df_cluster[columnas_scores_num].describe().T
        # Renombramos y ordenamos columnas para mayor claridad en español
        resumen_numerico = resumen_numerico[['count', 'mean', 'std', 'min', '50%', 'max']]
        resumen_numerico.columns = ['Válidos', 'Media (%)', 'Desv. Est.', 'Mínimo', 'Mediana', 'Máximo']
        
        display(HTML("<h4><i>Métricas de Riesgo Cuantitativo</i></h4>"))
        display(resumen_numerico.style.format(precision=2).set_properties(**{'text-align': 'center'}))
        
        # 4.2. Distribución de Categorías de Riesgo (Frecuencias)
        display(HTML("<h4><i>Distribución de Categorías de Riesgo (Frecuencia y Porcentaje)</i></h4>"))
        
        lista_tablas_cat = []
        for col_cat in columnas_scores_cat:
            # Contamos cuántos pacientes caen en cada nivel de riesgo
            conteo = df_cluster[col_cat].value_counts(dropna=True)
            porcentaje = df_cluster[col_cat].value_counts(normalize=True, dropna=True) * 100
            
            # Creamos un pequeño DataFrame para este score en específico
            df_cat = pd.DataFrame({
                'Categoría': conteo.index,
                f'{col_cat} (N)': conteo.values,
                f'{col_cat} (%)': porcentaje.values
            }).set_index('Categoría')
            
            lista_tablas_cat.append(df_cat)
        
        # Unimos las distribuciones de todos los scores en una sola tabla matriz
        resumen_categorico = pd.concat(lista_tablas_cat, axis=1).fillna(0)
        
        # Formateamos las columnas de conteo como enteros y los porcentajes con decimales
        formatos_columnas = {}
        for col in resumen_categorico.columns:
            if '(N)' in col:
                formatos_columnas[col] = "{:.0f}"
            else:
                formatos_columnas[col] = "{:.1f}%"
                
        display(resumen_categorico.style.format(formatos_columnas).set_properties(**{'text-align': 'center'}))
        display(HTML("<br><hr style='border-top: 1px dashed #ccc;'>"))


# GENERACIÓN DE TABLAS SCROLLEABLES
# Estilo CSS para forzar la barra de desplazamiento vertical (max-height de 300px)
estilo_scroll = [
    {'selector': '', 'props': [('max-height', '300px'), ('overflow-y', 'auto'), ('display', 'block')]}
]

columnas_visibles = [
    'id', 'cluster', 'edad_actual', 'colesterol_total', 'hdl', 'ta_sistolica',
    'Framingham_Mujer_%', 'Framingham_Mujer_Cat', 'SCORE2_Mujer_%', 'SCORE2_Mujer_Cat',
    'Framingham_Hombre_%', 'Framingham_Hombre_Cat', 'SCORE2_Hombre_%', 'SCORE2_Hombre_Cat'
]

for key, df_scored in resultados_completos.items():
    display(HTML(f"<h2 style='color:#2e6da4; margin-top:30px;'> Combinación: {key.upper()}</h2>"))
    
    # Dividir el output por cada clúster único dentro del archivo de k-medoids
    clusters_unicos = sorted(df_scored['cluster'].unique())
    
    for clus in clusters_unicos:
        df_cluster = df_scored[df_scored['cluster'] == clus][columnas_visibles]
        
        display(HTML(f"<h3><b>Clúster {clus}</b> — Total Pacientes calculados en esta vista: {len(df_cluster)}</h3>"))
        
        # Crear la tabla estilizada con scroll
        tabla_html = (df_cluster.style
                      .set_table_styles(estilo_scroll)
                      .highlight_null(color='#f2f2f2')
                      .set_properties(**{'text-align': 'center', 'padding': '6px'})
                      .to_html())
        
        display(HTML(tabla_html))

,Válidos,Media (%),Desv. Est.,Mínimo,Mediana,Máximo
Framingham_Mujer_%,90.00,5.26,3.70,0.50,4.47,24.90
SCORE2_Mujer_%,90.00,0.03,0.11,0.00,0.00,0.97
Framingham_Hombre_%,90.00,5.58,4.07,0.31,4.40,22.63
SCORE2_Hombre_%,90.00,0.02,0.06,0.00,0.00,0.47


,Framingham_Mujer_Cat (N),Framingham_Mujer_Cat (%),SCORE2_Mujer_Cat (N),SCORE2_Mujer_Cat (%),Framingham_Hombre_Cat (N),Framingham_Hombre_Cat (%),SCORE2_Hombre_Cat (N),SCORE2_Hombre_Cat (%)
Categoría,,,,,,,,
Bajo (<10%),84,93.3%,0,0.0%,82,91.1%,0,0.0%
Moderado (10-20%),5,5.6%,0,0.0%,7,7.8%,0,0.0%
Alto (>20%),1,1.1%,0,0.0%,1,1.1%,0,0.0%
Bajo (<2.5%),0,0.0%,88,97.8%,0,0.0%,88,97.8%
Bajo (<5%),0,0.0%,2,2.2%,0,0.0%,2,2.2%


,Válidos,Media (%),Desv. Est.,Mínimo,Mediana,Máximo
Framingham_Mujer_%,223.00,3.96,3.69,0.62,2.92,36.33
SCORE2_Mujer_%,223.00,0.01,0.04,0.00,0.00,0.33
Framingham_Hombre_%,223.00,4.21,3.47,0.46,3.18,26.74
SCORE2_Hombre_%,223.00,0.01,0.03,0.00,0.00,0.26


,Framingham_Mujer_Cat (N),Framingham_Mujer_Cat (%),SCORE2_Mujer_Cat (N),SCORE2_Mujer_Cat (%),Framingham_Hombre_Cat (N),Framingham_Hombre_Cat (%),SCORE2_Hombre_Cat (N),SCORE2_Hombre_Cat (%)
Categoría,,,,,,,,
Bajo (<10%),213,95.5%,0,0.0%,208,93.3%,0,0.0%
Moderado (10-20%),8,3.6%,0,0.0%,13,5.8%,0,0.0%
Alto (>20%),2,0.9%,0,0.0%,2,0.9%,0,0.0%
Bajo (<2.5%),0,0.0%,222,99.6%,0,0.0%,222,99.6%
Bajo (<5%),0,0.0%,1,0.4%,0,0.0%,1,0.4%


,Válidos,Media (%),Desv. Est.,Mínimo,Mediana,Máximo
Framingham_Mujer_%,223.00,3.96,3.69,0.62,2.92,36.33
SCORE2_Mujer_%,223.00,0.01,0.04,0.00,0.00,0.33
Framingham_Hombre_%,223.00,4.21,3.47,0.46,3.18,26.74
SCORE2_Hombre_%,223.00,0.01,0.03,0.00,0.00,0.26


...

,Framingham_Mujer_Cat (N),Framingham_Mujer_Cat (%),SCORE2_Mujer_Cat (N),SCORE2_Mujer_Cat (%),Framingham_Hombre_Cat (N),Framingham_Hombre_Cat (%),SCORE2_Hombre_Cat (N),SCORE2_Hombre_Cat (%)
Categoría,,,,,,,,
Bajo (<10%),84,93.3%,0,0.0%,82,91.1%,0,0.0%
Moderado (10-20%),5,5.6%,0,0.0%,7,7.8%,0,0.0%
Alto (>20%),1,1.1%,0,0.0%,1,1.1%,0,0.0%
Bajo (<2.5%),0,0.0%,88,97.8%,0,0.0%,88,97.8%
Bajo (<5%),0,0.0%,2,2.2%,0,0.0%,2,2.2%


,id,cluster,edad_actual,colesterol_total,hdl,ta_sistolica,Framingham_Mujer_%,Framingham_Mujer_Cat,SCORE2_Mujer_%,SCORE2_Mujer_Cat,Framingham_Hombre_%,Framingham_Hombre_Cat,SCORE2_Hombre_%,SCORE2_Hombre_Cat
0,1.000000,0,38.850000,148.220000,68.110000,113.000000,1.700000,Bajo (<10%),0.000000,Bajo (<2.5%),1.640000,Bajo (<10%),0.000000,Bajo (<2.5%)
1,2.000000,0,48.940000,172.220000,62.690000,97.000000,2.390000,Bajo (<10%),0.010000,Bajo (<2.5%),3.140000,Bajo (<10%),0.020000,Bajo (<2.5%)
2,4.000000,0,44.400000,209.370000,46.440000,128.000000,4.800000,Bajo (<10%),0.020000,Bajo (<2.5%),8.770000,Bajo (<10%),0.020000,Bajo (<2.5%)
3,13.000000,0,42.260000,177.630000,78.560000,116.000000,4.190000,Bajo (<10%),0.000000,Bajo (<2.5%),4.540000,Bajo (<10%),0.000000,Bajo (<2.5%)
4,32.000000,0,32.900000,177.630000,58.050000,119.000000,3.130000,Bajo (<10%),0.000000,Bajo (<2.5%),2.960000,Bajo (<10%),0.000000,Bajo (<2.5%)
5,33.000000,0,43.920000,169.890000,45.280000,117.000000,3.880000,Bajo (<10%),0.000000,Bajo (<2.5%),4.300000,Bajo (<10%),0.010000,Bajo (<2.5%)
6,39.000000,0,39.290000,203.950000,54.950000,122.000000,6.150000,Bajo (<10%),0.000000,Bajo (<2.5%),6.460000,Bajo (<10%),0.000000,Bajo (<2.5%)
7,42.000000,0,36.110000,178.790000,58.440000,131.000000,5.070000,Bajo (<10%),0.000000,Bajo (<2.5%),4.710000,Bajo (<10%),0.000000,Bajo (<2.5%)
8,53.000000,0,48.360000,209.750000,78.170000,106.000000,3.230000,Bajo (<10%),0.010000,Bajo (<2.5%),3.640000,Bajo (<10%),0.020000,Bajo (<2.5%)
9,56.000000,0,37.870000,178.410000,55.730000,130.000000,5.700000,Bajo (<10%),0.000000,Bajo (<2.5%),5.570000,Bajo (<10%),0.000000,Bajo (<2.5%)


,id,cluster,edad_actual,colesterol_total,hdl,ta_sistolica,Framingham_Mujer_%,Framingham_Mujer_Cat,SCORE2_Mujer_%,SCORE2_Mujer_Cat,Framingham_Hombre_%,Framingham_Hombre_Cat,SCORE2_Hombre_%,SCORE2_Hombre_Cat
132,5.000000,1,35.850000,176.470000,59.600000,115.000000,2.010000,Bajo (<10%),0.000000,Bajo (<2.5%),1.830000,Bajo (<10%),0.000000,Bajo (<2.5%)
133,8.000000,1,38.710000,174.150000,60.370000,130.000000,3.280000,Bajo (<10%),0.000000,Bajo (<2.5%),2.840000,Bajo (<10%),0.000000,Bajo (<2.5%)
134,10.000000,1,42.840000,214.400000,70.820000,115.000000,3.380000,Bajo (<10%),0.000000,Bajo (<2.5%),3.310000,Bajo (<10%),0.000000,Bajo (<2.5%)
135,11.000000,1,39.810000,157.120000,62.690000,96.000000,2.190000,Bajo (<10%),0.000000,Bajo (<2.5%),2.840000,Bajo (<10%),0.000000,Bajo (<2.5%)
136,14.000000,1,48.960000,144.740000,73.530000,89.000000,1.370000,Bajo (<10%),0.000000,Bajo (<2.5%),1.900000,Bajo (<10%),0.010000,Bajo (<2.5%)
137,15.000000,1,39.770000,176.860000,63.860000,90.000000,1.230000,Bajo (<10%),0.000000,Bajo (<2.5%),1.470000,Bajo (<10%),0.000000,Bajo (<2.5%)
138,18.000000,1,38.760000,189.240000,75.080000,105.000000,1.720000,Bajo (<10%),0.000000,Bajo (<2.5%),1.700000,Bajo (<10%),0.000000,Bajo (<2.5%)
139,19.000000,1,43.590000,194.660000,47.600000,96.000000,4.210000,Bajo (<10%),0.010000,Bajo (<2.5%),6.070000,Bajo (<10%),0.010000,Bajo (<2.5%)
140,26.000000,1,48.700000,109.520000,46.440000,96.000000,1.650000,Bajo (<10%),0.000000,Bajo (<2.5%),2.420000,Bajo (<10%),0.010000,Bajo (<2.5%)
141,28.000000,1,47.300000,177.630000,56.890000,98.000000,4.260000,Bajo (<10%),0.010000,Bajo (<2.5%),6.200000,Bajo (<10%),0.020000,Bajo (<2.5%)


,id,cluster,edad_actual,colesterol_total,hdl,ta_sistolica,Framingham_Mujer_%,Framingham_Mujer_Cat,SCORE2_Mujer_%,SCORE2_Mujer_Cat,Framingham_Hombre_%,Framingham_Hombre_Cat,SCORE2_Hombre_%,SCORE2_Hombre_Cat
0,5.000000,0,35.850000,176.470000,59.600000,115.000000,2.010000,Bajo (<10%),0.000000,Bajo (<2.5%),1.830000,Bajo (<10%),0.000000,Bajo (<2.5%)
1,8.000000,0,38.710000,174.150000,60.370000,130.000000,3.280000,Bajo (<10%),0.000000,Bajo (<2.5%),2.840000,Bajo (<10%),0.000000,Bajo (<2.5%)
2,10.000000,0,42.840000,214.400000,70.820000,115.000000,3.380000,Bajo (<10%),0.000000,Bajo (<2.5%),3.310000,Bajo (<10%),0.000000,Bajo (<2.5%)
3,11.000000,0,39.810000,157.120000,62.690000,96.000000,2.190000,Bajo (<10%),0.000000,Bajo (<2.5%),2.840000,Bajo (<10%),0.000000,Bajo (<2.5%)
4,14.000000,0,48.960000,144.740000,73.530000,89.000000,1.370000,Bajo (<10%),0.000000,Bajo (<2.5%),1.900000,Bajo (<10%),0.010000,Bajo (<2.5%)
5,15.000000,0,39.770000,176.860000,63.860000,90.000000,1.230000,Bajo (<10%),0.000000,Bajo (<2.5%),1.470000,Bajo (<10%),0.000000,Bajo (<2.5%)
6,18.000000,0,38.760000,189.240000,75.080000,105.000000,1.720000,Bajo (<10%),0.000000,Bajo (<2.5%),1.700000,Bajo (<10%),0.000000,Bajo (<2.5%)
7,19.000000,0,43.590000,194.660000,47.600000,96.000000,4.210000,Bajo (<10%),0.010000,Bajo (<2.5%),6.070000,Bajo (<10%),0.010000,Bajo (<2.5%)
8,26.000000,0,48.700000,109.520000,46.440000,96.000000,1.650000,Bajo (<10%),0.000000,Bajo (<2.5%),2.420000,Bajo (<10%),0.010000,Bajo (<2.5%)
9,28.000000,0,47.300000,177.630000,56.890000,98.000000,4.260000,Bajo (<10%),0.010000,Bajo (<2.5%),6.200000,Bajo (<10%),0.020000,Bajo (<2.5%)


,id,cluster,edad_actual,colesterol_total,hdl,ta_sistolica,Framingham_Mujer_%,Framingham_Mujer_Cat,SCORE2_Mujer_%,SCORE2_Mujer_Cat,Framingham_Hombre_%,Framingham_Hombre_Cat,SCORE2_Hombre_%,SCORE2_Hombre_Cat
326,1.000000,1,38.850000,148.220000,68.110000,113.000000,1.700000,Bajo (<10%),0.000000,Bajo (<2.5%),1.640000,Bajo (<10%),0.000000,Bajo (<2.5%)
327,2.000000,1,48.940000,172.220000,62.690000,97.000000,2.390000,Bajo (<10%),0.010000,Bajo (<2.5%),3.140000,Bajo (<10%),0.020000,Bajo (<2.5%)
328,4.000000,1,44.400000,209.370000,46.440000,128.000000,4.800000,Bajo (<10%),0.020000,Bajo (<2.5%),8.770000,Bajo (<10%),0.020000,Bajo (<2.5%)
329,13.000000,1,42.260000,177.630000,78.560000,116.000000,4.190000,Bajo (<10%),0.000000,Bajo (<2.5%),4.540000,Bajo (<10%),0.000000,Bajo (<2.5%)
330,32.000000,1,32.900000,177.630000,58.050000,119.000000,3.130000,Bajo (<10%),0.000000,Bajo (<2.5%),2.960000,Bajo (<10%),0.000000,Bajo (<2.5%)
331,33.000000,1,43.920000,169.890000,45.280000,117.000000,3.880000,Bajo (<10%),0.000000,Bajo (<2.5%),4.300000,Bajo (<10%),0.010000,Bajo (<2.5%)
332,39.000000,1,39.290000,203.950000,54.950000,122.000000,6.150000,Bajo (<10%),0.000000,Bajo (<2.5%),6.460000,Bajo (<10%),0.000000,Bajo (<2.5%)
333,42.000000,1,36.110000,178.790000,58.440000,131.000000,5.070000,Bajo (<10%),0.000000,Bajo (<2.5%),4.710000,Bajo (<10%),0.000000,Bajo (<2.5%)
334,53.000000,1,48.360000,209.750000,78.170000,106.000000,3.230000,Bajo (<10%),0.010000,Bajo (<2.5%),3.640000,Bajo (<10%),0.020000,Bajo (<2.5%)
335,56.000000,1,37.870000,178.410000,55.730000,130.000000,5.700000,Bajo (<10%),0.000000,Bajo (<2.5%),5.570000,Bajo (<10%),0.000000,Bajo (<2.5%)
